# 🐛 Debugging — Part 2: Intermediate Techniques

> **Module:** 01 — Programming Best Practices  
> **Series:** Debugging in VS Code (2 of 3)  
> **Covers:** Conditional breakpoints, logpoints, `launch.json`, CLI arguments, call stack

**Previous ←** [01_fundamentals.ipynb](01_fundamentals.ipynb)  
**Next →** [03_advanced.ipynb](03_advanced.ipynb)

---

## 1 — Conditional Breakpoints

A regular breakpoint pauses **every time** the line is hit. In a loop with 10 000 iterations, that's useless.  
A **conditional breakpoint** pauses only when a Python expression evaluates to `True`.

### How to set one

1. Right-click the gutter → **"Add Conditional Breakpoint..."**
2. Choose **Expression** and type any valid Python: `i == 50`, `name == "Charlie"`, `amount > 500`
3. The breakpoint appears as a red dot with an **=** sign

![image.png](attachment:image.png)

### Example — find a specific transaction

Open `debug_example.py` and locate the `audit_transactions` function (Part 2 section):

1. Set a **conditional breakpoint** on the line `total += t["amount"]` with the expression:  
   `t["user"] == "charlie" and t["amount"] > 1000`
2. Press `F5`. The loop runs through alice, bob, etc. — the debugger **does not stop**.
3. When it reaches Charlie's $1 500 withdrawal, the condition is `True` → the debugger **pauses**.
4. Inspect `t` in the Variables panel to see the full transaction dict.

This saves you from stepping through every iteration manually.

### Hit Count breakpoints

A variant: pause after the line has been reached **N times**.

- Right-click gutter → **"Add Conditional Breakpoint..."** → **Hit Count** → enter `5`
- The breakpoint fires on the 5th hit — useful when the bug appears deep into a long loop.

---

## 2 — Logpoints

A **logpoint** prints a message to the Debug Console **without pausing** execution.  
Think of it as a `print()` statement that you don't have to add to (or remove from) your code.

### How to set one

1. Right-click the gutter → **"Add Logpoint..."**
2. Type a message with expressions in curly braces:  
   `Processing {user}: ${amount:.2f} | running total: ${total:.2f}`
3. The logpoint appears as a **diamond ◆** instead of a circle
4. Output goes to the **Debug Console** panel (`Ctrl+Shift+Y`)

![image.png](attachment:image.png)

> **Logpoints are the professional replacement for `print()` debugging.**  
> They don't modify your source, they disappear when the session ends, and they work on code you can't (or shouldn't) edit.

In [ ]:
# ============================================================
# Companion code — try conditional breakpoints and logpoints
# against this function.
#
# Suggested experiments:
#   • Conditional breakpoint on `suspicious.append(t)` with
#     condition:  t["amount"] > 500
#   • Logpoint on `total += t["amount"]` with message:
#     Processing {t['user']}: ${t['amount']:.2f} | total so far: ${total:.2f}
# ============================================================

def audit_transactions(transactions: list[dict]) -> dict:
    total = 0.0
    suspicious: list[dict] = []
    by_user: dict[str, float] = {}

    for t in transactions:
        total += t["amount"]
        by_user[t["user"]] = by_user.get(t["user"], 0) + t["amount"]
        if t["amount"] > 500:
            suspicious.append(t)

    return {"total": total, "suspicious": suspicious, "by_user": by_user}


transactions = [
    {"user": "alice",   "amount": 120.50,  "type": "purchase"},
    {"user": "bob",     "amount": 890.00,  "type": "transfer"},
    {"user": "alice",   "amount": 45.99,   "type": "purchase"},
    {"user": "charlie", "amount": 1500.00, "type": "withdrawal"},
    {"user": "bob",     "amount": 200.00,  "type": "purchase"},
    {"user": "alice",   "amount": 675.25,  "type": "transfer"},
    {"user": "diana",   "amount": 50.00,   "type": "deposit"},
    {"user": "charlie", "amount": 2200.00, "type": "transfer"},
]

report = audit_transactions(transactions)
print(f"Total: ${report['total']:.2f}")
print(f"Suspicious: {len(report['suspicious'])}")
for s in report["suspicious"]:
    print(f"  \u26a0 {s['user']}: ${s['amount']:.2f}")

---

## 3 — `launch.json` Configuration

Without a `launch.json`, VS Code debugs the currently open file with default settings.  
With a `launch.json`, you save **reusable configurations**: fixed entry points, command-line arguments, environment variables, pre-launch tasks, etc.

### Creating the file

1. Open the Debug panel (`Ctrl+Shift+D`)
2. Click **"create a launch.json file"** → select **Python Debugger** → **Python File**
3. VS Code creates `.vscode/launch.json`

### Anatomy of a configuration

```json
{
    "name": "Debug Main Script",
    "type": "debugpy",
    "request": "launch",
    "program": "${workspaceFolder}/src/main.py",
    "args": ["--input", "data.csv", "--verbose"],
    "cwd": "${workspaceFolder}",
    "env": {"ENVIRONMENT": "development"},
    "console": "integratedTerminal",
    "justMyCode": true
}
```

| Property | Purpose |
|----------|--------|
| `name` | Label shown in the debug dropdown |
| `type` | Always `"debugpy"` for Python |
| `request` | `"launch"` (start new process) or `"attach"` (connect to running one) |
| `program` | Script to run. `${file}` = currently open file |
| `args` | Command-line arguments as a list |
| `env` | Environment variables injected into the process |
| `justMyCode` | `true` skips library internals; `false` lets you step into pandas, requests, etc. |
| `module` | Run as `python -m <module>` instead of a file |
| `console` | `"integratedTerminal"` (normal) or `"internalConsole"` |

### Common configurations

Below is a full `launch.json` with the configurations you'll use most often.  
Copy it to `.vscode/launch.json` in your project and adapt the paths.

```json
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "Python: Current File",
            "type": "debugpy",
            "request": "launch",
            "program": "${file}",
            "console": "integratedTerminal",
            "justMyCode": true
        },
        {
            "name": "Python: With Arguments",
            "type": "debugpy",
            "request": "launch",
            "program": "${workspaceFolder}/src/cli.py",
            "args": ["--input", "data.csv", "--threshold", "0.75"],
            "console": "integratedTerminal"
        },
        {
            "name": "Python: Pytest",
            "type": "debugpy",
            "request": "launch",
            "module": "pytest",
            "args": ["tests/", "-v", "--tb=short"],
            "justMyCode": false
        },
        {
            "name": "Python: Specific Test",
            "type": "debugpy",
            "request": "launch",
            "module": "pytest",
            "args": ["tests/test_pipeline.py::test_clean_data", "-v", "-s"],
            "justMyCode": false
        },
        {
            "name": "Python: Module",
            "type": "debugpy",
            "request": "launch",
            "module": "mypackage.main",
            "cwd": "${workspaceFolder}"
        }
    ]
}
```

---

## 4 — Debugging Scripts with Command-Line Arguments

Production scripts often use `argparse` or `sys.argv`. If you just press `F5`, the script receives **no arguments** and typically crashes or uses defaults.

The solution is simple: add an `"args"` list in `launch.json`.

```json
"args": ["--input", "data.csv", "--threshold", "500", "--verbose"]
```

Each element maps to one token on the command line:

```bash
python cli_tool.py --input data.csv --threshold 500 --verbose
```

Set a breakpoint right after `parser.parse_args()` and inspect the `args` namespace to verify everything was parsed correctly.

In [ ]:
# ============================================================
# Example CLI script — save as cli_tool.py
# Configure launch.json args:
#   ["--input", "sales.csv", "--threshold", "500", "--verbose"]
# ============================================================

import argparse


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Analyze sales data")
    parser.add_argument("--input", required=True, help="Input file")
    parser.add_argument("--threshold", type=float, default=100.0)
    parser.add_argument("--verbose", action="store_true")
    return parser.parse_args()


def analyze(file_path: str, threshold: float, verbose: bool) -> None:
    records = [
        {"product": "Widget A", "revenue": 1200.0},
        {"product": "Widget B", "revenue": 350.0},
        {"product": "Widget C", "revenue": 89.5},
        {"product": "Widget D", "revenue": 2100.0},
    ]
    if verbose:
        print(f"Analyzing {file_path}, threshold=${threshold:.2f}")

    alerts = [r for r in records if r["revenue"] > threshold]
    print(f"Records above ${threshold:.2f}: {len(alerts)}")
    for a in alerts:
        print(f"  \u2192 {a['product']}: ${a['revenue']:.2f}")


if __name__ == "__main__":
    args = parse_args()  # \u2190 set breakpoint here to inspect parsed args
    analyze(args.input, args.threshold, args.verbose)

---

## 5 — The Call Stack & Recursive Functions

The **CALL STACK** panel (Debug sidebar) shows the chain of function calls that led to the current line. Every function call pushes a new frame; every return pops one.

### Why it matters

- See exactly **how deep** the recursion has gone
- **Click on any frame** to jump to that level — the Variables panel updates accordingly
- Compare the same parameter (`n`) across different recursion depths
- Detect infinite recursion before a `RecursionError` crash

### Walkthrough — factorial

Open `debug_example.py` and set a breakpoint on `return n * factorial(n - 1)` in the `factorial` function (Part 2 section).

Run the file with `F5`. When the breakpoint hits:

```
CALL STACK:
  factorial  n=1   ← current frame (deepest)
  factorial  n=2
  factorial  n=3
  factorial  n=4
  factorial  n=5   ← first call
  <module>
```

Click on the `n=3` frame: the Variables panel shows `n = 3`. Click on `n=5`: it shows `n = 5`. You're inspecting different points of the same recursion simultaneously.

The code cell below has additional recursive examples (fibonacci, merge_sort) you can practice with.

In [ ]:
# ============================================================
# Recursive examples — step through these with the debugger
# to see the Call Stack grow and shrink.
# ============================================================

def factorial(n: int) -> int:
    if n <= 1:
        return 1
    return n * factorial(n - 1)   # breakpoint here — watch the stack


def fibonacci(n: int) -> int:
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)  # two recursive calls per frame


def merge_sort(arr: list[int]) -> list[int]:
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    merged, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    return merged + left[i:] + right[j:]


print(f"5! = {factorial(5)}")
print(f"fib(7) = {fibonacci(7)}")
print(f"sorted: {merge_sort([38, 27, 43, 3, 9, 82])}")

---

## Recap — Part 2

| Concept | Key takeaway |
|---------|-------------|
| **Conditional breakpoints** | Pause only when an expression is true — essential for loops |
| **Hit Count breakpoints** | Pause on the Nth hit |
| **Logpoints** | Print to Debug Console without pausing or editing code |
| **`launch.json`** | Save reusable debug configs: entry point, args, env vars |
| **CLI arguments** | Pass via the `"args"` list in `launch.json` |
| **Call Stack** | Navigate recursion levels; click a frame to inspect its variables |

**Next →** [03_advanced.ipynb](03_advanced.ipynb) — Exception debugging, multi-file stepping, test debugging, remote attach, and the Debug Console.